### Supervisor

Multi-Agent 시스템의 구성 방식 중 하나인 Supervisor 패턴을 살펴본다.

In [1]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

MODEL_NAME = "gemini-3.6-flash"
llm = ChatGoogleGenerativeAI(model=MODEL_NAME)
search_llm = llm.bind_tools([{"google_search": {}}])

### Multi-Agent란

Multi-Agent 시스템은 하나의 Agent가 모든 작업을 처리하는 대신, 서로 다른 역할을 맡은 여러 Agent가 협력하여 하나의 목표를 달성하는 구조이다. 각 Agent는 역할에 맞는 Prompt, Tool, Context를 독립적으로 가질 수 있으며, Agent 사이의 작업 전달과 결과 공유를 통해 복잡한 작업을 나누어 처리한다.

### Multi-Agent를 고려하는 이유

하나의 Agent가 너무 많은 역할, Tool, Context를 담당하면 지시 충돌과 Tool 선택 오류가 늘어날 수 있다. 역할별 Agent로 분리하면 각 Agent가 자신의 작업에 필요한 지시와 정보에 집중할 수 있다.

| 이점 | 설명 |
|------|------|
| 역할과 Context 분리 | 각 Agent가 담당 작업에 필요한 정보에 집중 |
| Tool과 권한 분리 | Agent별로 필요한 Tool만 제공 |
| 독립적인 설계와 평가 | 역할마다 Prompt, 모델, 평가 기준을 다르게 설정 |
| 동적인 작업 위임 | 요청과 중간 결과에 따라 적절한 Agent를 선택 |

역할이 단순하고 실행 순서가 고정되어 있다면 단일 Agent나 일반 Workflow가 더 적합하다. Multi-Agent는 모델 호출 수, 지연 시간, 비용과 실패 지점을 증가시키므로 역할 분리와 동적 위임의 이점이 충분할 때 사용한다.

### Supervisor 패턴

Supervisor 패턴은 중앙의 Supervisor가 작업을 분석하여 적절한 Subagent에 분배하고, 각 결과를 종합해 최종 응답을 만드는 구조이다.

```text
사용자 → Supervisor Agent
             ├─→ Research Agent
             ├─→ Writer Agent
             └─→ Reviewer Agent
```

### Supervisor 구현

보고서 작성 과정을 자료 조사, 작성, 검토로 나눈다. `create_agent`로 만든 각 Subagent를 `@tool` 함수로 감싸 Supervisor의 Tool로 등록한다. 각 Tool은 작업 설명을 Subagent에 전달하고 마지막 응답만 Supervisor에 반환한다. Checkpointer는 전체 대화를 관리하는 최상위 Supervisor에만 설정한다.

In [2]:
researcher_agent = create_agent(
    model=search_llm,
    tools=[],
    system_prompt=(
        "너는 자료 조사 전문가다. 요청받은 주제를 Google Search로 조사해. "
        "보고서 작성에 필요한 핵심 사실과 근거를 정리하고, 각 내용의 출처를 함께 제시해."
    ),
    name="researcher",
)

writer_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "너는 보고서 작성 전문가다. 전달받은 조사 결과만을 근거로 보고서를 작성해. "
        "보고서는 제목, 개요, 본문, 결론 순서로 구성하고 출처를 유지해."
    ),
    name="writer",
)

reviewer_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "너는 보고서 검토 전문가다. 전달받은 보고서의 구조, 가독성, 내용의 일관성, "
        "출처 표기를 검토해. 수정이 필요하면 구체적인 수정 사항을 제시하고, "
        "문제가 없으면 '승인'이라고 명시해."
    ),
    name="reviewer",
)

In [3]:
@tool
def research(query: str) -> str:
    """주제에 관한 최신 자료와 출처를 조사한다."""
    result = researcher_agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return result["messages"][-1].text


@tool
def write_report(query: str) -> str:
    """조사 결과를 바탕으로 보고서를 작성한다."""
    result = writer_agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return result["messages"][-1].text


@tool
def review_report(query: str) -> str:
    """보고서를 검토하고 수정 사항이나 승인 여부를 반환한다."""
    result = reviewer_agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return result["messages"][-1].text

In [4]:
supervisor_agent = create_agent(
    model=llm,
    tools=[research, write_report, review_report],
    system_prompt=(
        "너는 보고서 작성 팀의 Supervisor다. "
        "사용자의 요청을 분석하고 필요한 작업을 적절한 전문가에게 위임해. "
        "작업을 위임할 때는 해당 전문가에게 필요한 맥락을 충분히 전달해. "
        "전문가의 결과를 검토하고 필요한 경우 추가 작업을 요청해. "
        "근거와 출처가 포함된 완성도 높은 보고서를 최종 응답으로 제공해. "
        "중간 작업 과정은 사용자에게 설명하지 마."
    ),
    checkpointer=InMemorySaver(),
)

### 실행

In [5]:
config = {"configurable": {"thread_id": "report-session-1"}}

for event in supervisor_agent.stream(
    {"messages": [HumanMessage(content="AI Agent 기술 동향에 대한 보고서를 작성해줘.")]},
    config=config,
    stream_mode="updates",
):
    for node_name, state in event.items():
        message = state["messages"][-1]
        print("=" * 60)
        print(f"Node: {node_name}")
        if getattr(message, "tool_calls", None):
            print(f"Tool calls: {[call['name'] for call in message.tool_calls]}")
        if message.text:
            print(message.text[:500])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Node: model
Tool calls: ['research']
Node: tools
**AI Agent 기술 동향 및 자율형 아키텍처·멀티에이전트 산업 적용 사례 보고서**

---

### 1. 개요: 생성형 AI에서 에이전틱 AI(Agentic AI)로의 전환

2024년 이후 인공지능 분야의 핵심 트렌드는 단순 질문-답변 형태의 대화형 LLM(Chatbot)에서, 스스로 목표를 세우고 외부 도구를 활용해 복잡한 과업을 완료하는 **자율형 AI 에이전트(Agentic AI)**로 빠르게 변화하고 있습니다.

가트너(Gartner)는 2026년까지 엔터프라이즈 애플리케이션의 **40% 이상**에 업무 특화형 AI 에이전트가 탑재될 것으로 전망했으며, 멕킨지(McKinsey) 조사에 따르면 주요 기업의 23%가 이미 에이전틱 시스템을 확장 적용 중인 것으로 나타났습니다.

---

### 2. 자율형 AI 에이전트의 핵심 아키텍처 (Autonomous AI Architecture)

LLM 기반 자율형 에이전트는 대규모 언어 모델을 **'두뇌(Brain)'**로 삼
Node: model
Tool calls: ['write_report']
Node: tools
제공된 AI Agent 기술 조사 항목(기술 동향, 아키텍처, 멀티에이전트, 산업별 적용 사례, 과제 및 전망)을 종합적으로 정리한 **기술 동향 보고서**입니다.

---

# [종합 기술 보고서] AI Agent 기술 동향, 아키텍처 및 산업 적용 전망

## 1. 개요 (Executive Summary)

*   **배경:** 생성형 AI 기술이 단순 텍스트 생성(Chatbot/Copilot) 단계를 넘어, 스스로 목표를 설정하고 계획을 수립하며 외부 도구를 활용해 복잡한 과업을 수행하는 **'자율형 AI 에이전트(AI Agent)'**로 급격히 진화하고 있음.
*   **목적:** AI Agent의 핵심 아키텍처 구성 요소, 멀티에이전트 협업 체계, 산업별 도입 현황 및 주요 한계점을 분석하고 

### Supervisor의 동작 방식

이 예제의 Supervisor도 필요한 작업을 판단하고, Tool을 실행하고, 결과를 관찰해 다음 행동을 결정하는 ReAct 방식으로 동작한다. 여기서 Tool은 독립적인 지시와 Context를 가진 Subagent를 호출한다. 예를 들어 자료 조사가 필요하면 `research`를 호출하고, 반환된 결과를 바탕으로 보고서 작성이나 검토를 위임한 뒤 최종 응답을 만든다.